In [1]:
#!/usr/bin/env python3
"""
DGH-XH: Beijing Multi-Site Temperature Forecasting
====================================================
Dataset : UCI Beijing Multi-Site Air Quality Data (same CSV files)
          Kaggle: sid321axn/beijing-multisite-airquality-data-set
Target  : TEMP (°C) — temperature forecasting, NOT air quality
Stations: 12 (KNN k=4 graph)
Split   : test from 2016-09-01
KEY DIFFERENCE FROM BEIJING AQI NOTEBOOK:
  TARGET_COL = "temp" instead of "pm25"
  Features exclude pm25, use only meteorological variables
  Tail events = heatwaves / cold snaps (operationally meaningful)
"""

# ============================================================
# CELL 1 — IMPORTS  (identical)
# ============================================================
import os, warnings, joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)
import xgboost as xgb
from xgboost import XGBRegressor
from IPython.display import display
warnings.filterwarnings("ignore")

# ============================================================
# CELL 2 — CONFIG  (only TARGET_COL and FEATURES change)
# ============================================================
BEIJING_DATA_DIR = "/kaggle/input/datasets/sid321axn/beijing-multisite-airquality-data-set"
OUT_DIR          = "dghxh_beijing_temp"
SPLIT_TIME       = pd.Timestamp("2016-09-01 00:00:00")
L                = 24
H_LIST           = [1, 3, 6, 12, 24]
TAU_LIST         = [0, 1, 2, 3, 4, 6]
VALID_FRAC       = 0.20
GRAPH_K          = 4
RUN_TUNED        = True
JSO_POP          = 6   # 48 evals (was 180): saves ~1.5h/horizon
JSO_ITERS        = 8   # reduced from 15
RANDOM_SEED      = 1

TARGET_COL = "temp"        # ← KEY DIFFERENCE: temperature, not PM2.5
# Features: temperature + pressure + dewpoint + wind (no PM2.5)
# Temperature is both input (lag features) and target — standard in NWP
FEATURES   = ["temp", "pres", "dewp", "u_wind", "v_wind"]

STATION_COORDS = {
    "Aotizhongxin":  (40.00, 116.41),
    "Changping":     (40.22, 116.23),
    "Dingling":      (40.29, 116.22),
    "Dongsi":        (39.93, 116.42),
    "Guanyuan":      (39.93, 116.34),
    "Gucheng":       (39.91, 116.18),
    "Huairou":       (40.38, 116.63),
    "Nongzhanguan":  (39.94, 116.46),
    "Shunyi":        (40.13, 116.65),
    "Tiantan":       (39.88, 116.41),
    "Wanliu":        (39.99, 116.30),
    "Wanshouxigong": (39.88, 116.36),
}

# ============================================================
# CELL 3 — WIND CONVERSION HELPER (identical to Beijing AQI)
# ============================================================
def met_wind_to_uv(wd_deg_series, wspd_series):
    """
    Meteorological wind direction (direction wind comes FROM, degrees
    clockwise from N) + speed → eastward (u) / northward (v) components.
    Physical basis for temperature: warm/cold air masses advect in the
    wind motion direction, identical to the pollutant advection mechanism.
    """
    wd_deg = pd.to_numeric(wd_deg_series, errors="coerce").fillna(0.0)
    wspd   = pd.to_numeric(wspd_series, errors="coerce").fillna(0.0)
    theta  = np.radians(wd_deg.to_numpy())
    u = (-np.sin(theta) * wspd.to_numpy()).astype(np.float32)
    v = (-np.cos(theta) * wspd.to_numpy()).astype(np.float32)
    return u, v

# ============================================================
# CELL 4 — DATA LOADER (identical loading, different FEATURES used)
# ============================================================
def load_beijing_temp():
    dfs = []
    wd_str_to_deg = {
        "N":0.0,"NNE":22.5,"NE":45.0,"ENE":67.5,
        "E":90.0,"ESE":112.5,"SE":135.0,"SSE":157.5,
        "S":180.0,"SSW":202.5,"SW":225.0,"WSW":247.5,
        "W":270.0,"WNW":292.5,"NW":315.0,"NNW":337.5,
    }
    for fname in sorted(os.listdir(BEIJING_DATA_DIR)):
        if not fname.endswith(".csv"): continue
        parts = fname.replace(".csv","").split("_")
        station = parts[2] if len(parts)>=3 else fname.replace(".csv","")
        if station not in STATION_COORDS: continue

        df = pd.read_csv(os.path.join(BEIJING_DATA_DIR, fname))
        df["timestamp"] = pd.to_datetime(df[["year","month","day","hour"]])

        if df["wd"].dtype == object:
            df["wd_deg"] = df["wd"].map(wd_str_to_deg).fillna(0.0)
        else:
            df["wd_deg"] = pd.to_numeric(df["wd"], errors="coerce").fillna(0.0)

        u, v = met_wind_to_uv(df["wd_deg"], df["WSPM"])
        df["u_wind"] = u; df["v_wind"] = v

        df["station_id"] = station
        df["lat"] = STATION_COORDS[station][0]
        df["lon"] = STATION_COORDS[station][1]
        df["temp"] = pd.to_numeric(df["TEMP"], errors="coerce")
        df["pres"] = pd.to_numeric(df["PRES"], errors="coerce")
        df["dewp"] = pd.to_numeric(df["DEWP"], errors="coerce")

        dfs.append(df[["timestamp","station_id","lat","lon",
                        "temp","pres","dewp","u_wind","v_wind"]])

    df_all = pd.concat(dfs, ignore_index=True).sort_values(
        ["timestamp","station_id"]).reset_index(drop=True)
    print(f"Loaded {len(df_all):,} rows, "
          f"{df_all.station_id.nunique()} stations")
    return df_all

# ============================================================
# ALL SHARED HELPER FUNCTIONS
# (Copy from Beijing AQI notebook — identical, included here
#  so this file is self-contained and runnable independently)
# ============================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1,lon1,lat2,lon2 = map(np.radians,[lat1,lon1,lat2,lon2])
    dlat=lat2-lat1; dlon=lon2-lon1
    a = np.sin(dlat/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2.0*R*np.arcsin(np.sqrt(a))

def bearing_radians(lat1, lon1, lat2, lon2):
    lat1,lon1,lat2,lon2 = map(np.radians,[lat1,lon1,lat2,lon2])
    dlon=lon2-lon1
    y=np.sin(dlon)*np.cos(lat2)
    x=np.cos(lat1)*np.sin(lat2)-np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    return np.arctan2(y,x)

def flatten_window_per_node(X):
    S,Lx,N,F=X.shape; return X.transpose(0,2,1,3).reshape(S*N,Lx*F)

def compute_tail_metrics(y_true, y_pred, percentiles=(90,95,99)):
    rows=[]
    for p in percentiles:
        thr=np.percentile(y_true,p); idx=y_true>=thr
        if idx.sum()==0: rows.append((p,thr,np.nan,np.nan,0))
        else: rows.append((p,thr,
            mean_absolute_error(y_true[idx],y_pred[idx]),
            np.sqrt(mean_squared_error(y_true[idx],y_pred[idx])),int(idx.sum())))
    return rows

def compute_mfb_nmse(y_true, y_pred):
    yt=np.asarray(y_true,dtype=float); yp=np.asarray(y_pred,dtype=float)
    return np.mean(2.0*(yp-yt)/(yp+yt+1e-8)), np.sum((yp-yt)**2)/(np.sum(yp*yt)+1e-8)

def build_fill_values_from_train_timeline(X):
    fv=np.nanmedian(X,axis=0); gf=np.nanmedian(X.reshape(-1,X.shape[-1]),axis=0)
    gf=np.where(np.isnan(gf),0.0,gf)
    for n in range(fv.shape[0]):
        for f in range(fv.shape[1]):
            if np.isnan(fv[n,f]): fv[n,f]=gf[f]
    return np.where(np.isnan(fv),0.0,fv).astype(np.float32)

def impute_windows(X, fill_values):
    X_imp=X.copy().astype(np.float32); S,Lx,N,F=X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                ser=pd.Series(X_imp[s,:,n,f],dtype="float32").ffill().bfill()
                arr=ser.to_numpy(dtype=np.float32)
                if np.isnan(arr).any(): arr=np.where(np.isnan(arr),fill_values[n,f],arr)
                X_imp[s,:,n,f]=arr
    return X_imp

def build_nodes_from_coords(station_ids, coord_dict):
    return pd.DataFrame([{"station_id":sid,"lat":coord_dict[sid][0],
                           "lon":coord_dict[sid][1],"node_id":i}
                          for i,sid in enumerate(station_ids)])

def build_edges_from_nodes(nodes_df, k=4):
    coords=nodes_df[["lat","lon"]].to_numpy(dtype=float); N=len(coords)
    D=np.zeros((N,N))
    for i in range(N): D[i,:]=haversine_km_vec(coords[i,0],coords[i,1],coords[:,0],coords[:,1])
    sigma=np.median(D[D>0]); edges=[]
    for i in range(N):
        for j in np.argsort(D[i])[1:min(k+1,N)]:
            edges.append((i,j,float(np.exp(-(D[i,j]**2)/(2*sigma**2))),float(D[i,j])))
    return pd.DataFrame(edges,columns=["src","dst","w_dist","dist_km"])

def build_static_adj(nodes_df, k=4):
    coords=nodes_df[["lat","lon"]].to_numpy(dtype=float); N=len(coords)
    D=np.zeros((N,N))
    for i in range(N): D[i,:]=haversine_km_vec(coords[i,0],coords[i,1],coords[:,0],coords[:,1])
    sigma=np.median(D[D>0]); A=np.zeros((N,N),dtype=np.float32)
    for i in range(N):
        for j in np.argsort(D[i])[1:min(k+1,N)]:
            A[i,j]=np.exp(-(D[i,j]**2)/(2*sigma**2))
    rs=A.sum(axis=1,keepdims=True); return np.divide(A,rs,out=np.zeros_like(A),where=rs>0)

def build_dynamic_adj(u_t, v_t, ctx, alpha=4.0, eps=0.05):
    theta_w=np.arctan2(v_t[ctx["src"]],u_t[ctx["src"]])
    gate=eps+(1-eps)/(1+np.exp(-alpha*np.cos(theta_w-ctx["edge_bearing"])))
    w=ctx["w_dist"]*gate; A=np.zeros((ctx["N"],ctx["N"]),dtype=np.float32)
    A[ctx["src"],ctx["dst"]]=w.astype(np.float32)
    rs=A.sum(axis=1,keepdims=True); return np.divide(A,rs,out=np.zeros_like(A),where=rs>0)

def make_graph_features_dynamic(X, ctx, tau=0, alpha=4.0, eps=0.05):
    S,Lx,N,F=X.shape; Z=np.zeros((S,N,2*F),dtype=np.float32)
    for s in range(S):
        x_now=X[s,-1]; x_lag=X[s,-1-tau] if tau>0 else x_now
        A=build_dynamic_adj(x_now[:,ctx["u_idx"]],x_now[:,ctx["v_idx"]],ctx,alpha,eps)
        Z[s]=np.concatenate([x_now,A@x_lag],axis=-1)
    return Z

def make_graph_features_static(X, A_static, tau=0):
    S,Lx,N,F=X.shape; Z=np.zeros((S,N,2*F),dtype=np.float32)
    for s in range(S):
        x_now=X[s,-1]; x_lag=X[s,-1-tau] if tau>0 else x_now
        Z[s]=np.concatenate([x_now,A_static@x_lag],axis=-1)
    return Z

def summarize_regression(yt, yp):
    return {"MAE":float(mean_absolute_error(yt,yp)),
            "RMSE":float(np.sqrt(mean_squared_error(yt,yp))),
            "R2":float(r2_score(yt,yp))}

def train_val_split_time_order(X,Y,M,times,frac=0.2):
    n=X.shape[0]; v=max(1,int(np.ceil(frac*n)))
    tr={"X":X[:n-v],"Y":Y[:n-v],"M":M[:n-v],"times":times[:n-v]}
    va={"X":X[n-v:],"Y":Y[n-v:],"M":M[n-v:],"times":times[n-v:]}
    return tr, va

def fit_flat_regressor(model, Xtr,Ytr,Mtr, Xte,Yte,Mte):
    Xt2=flatten_window_per_node(Xtr); Xe2=flatten_window_per_node(Xte)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1)
    mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    model.fit(Xt2[mt],yt[mt]); return {"y_true":ye[me],"y_pred":model.predict(Xe2)[me],"model":model}

def fit_static_graph_regressor(model, Xtr,Ytr,Mtr, Xte,Yte,Mte, A_static, tau):
    Zt=make_graph_features_static(Xtr,A_static,tau); Ze=make_graph_features_static(Xte,A_static,tau)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1)
    mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    Xt2=Zt.reshape(-1,Zt.shape[-1]); Xe2=Ze.reshape(-1,Ze.shape[-1])
    model.fit(Xt2[mt],yt[mt]); return {"y_true":ye[me],"y_pred":model.predict(Xe2)[me],"model":model}

def fit_huber_graph(Xtr,Ytr,Mtr, Xte,Yte,Mte, ctx,tau=0,alpha=4.0,eps=0.05,huber_alpha=1e-4):
    Zt=make_graph_features_dynamic(Xtr,ctx,tau,alpha,eps)
    Ze=make_graph_features_dynamic(Xte,ctx,tau,alpha,eps)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1)
    mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    Xt2=Zt.reshape(-1,Zt.shape[-1]); Xe2=Ze.reshape(-1,Ze.shape[-1])
    huber=Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,alpha=huber_alpha,max_iter=500))])
    huber.fit(Xt2[mt],yt[mt])
    ptr=huber.predict(Xt2); pte=huber.predict(Xe2)
    return {"y_true":ye[me],"y_pred":pte[me],"pred_train_all":ptr,"pred_test_all":pte,
            "y_train_all":yt,"y_test_all":ye,"mask_train":mt,"mask_test":me,"model":huber}

def fit_huber_hybrid(Xtr,Ytr,Mtr, Xte,Yte,Mte, ctx,tau=0,alpha=4.0,eps=0.05,
                     huber_alpha=1e-4,xgb_params=None,num_boost_round=400,
                     seed=42,w_mid=2.0,w_danger=5.0,w_tail=10.0):
    base=fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,ctx,tau,alpha,eps,huber_alpha)
    yt=base["y_train_all"]; mt=base["mask_train"]; me=base["mask_test"]
    res=yt-base["pred_train_all"]
    Xg_tr=np.asarray(flatten_window_per_node(Xtr),dtype=np.float32)
    Xg_te=np.asarray(flatten_window_per_node(Xte),dtype=np.float32)
    t90,t95,t99=np.percentile(yt[mt],[90,95,99])
    w=np.ones_like(yt[mt],dtype=np.float32)
    w[yt[mt]>=t90]=w_mid; w[yt[mt]>=t95]=w_danger; w[yt[mt]>=t99]=w_tail
    dtrain=xgb.DMatrix(Xg_tr[mt],label=res[mt],weight=w)
    if xgb_params is None:
        xgb_params={"objective":"reg:pseudohubererror","max_depth":6,"eta":0.05,
                    "subsample":0.80,"colsample_bytree":0.80,"lambda":1.0,
                    "tree_method":"hist","seed":seed,"verbosity":0}
    booster=xgb.train(xgb_params,dtrain,num_boost_round=int(num_boost_round))
    res_te=booster.predict(xgb.DMatrix(Xg_te))
    final=base["pred_test_all"].copy(); final[me]=final[me]+res_te[me]
    return {"y_true":base["y_test_all"][me],"y_pred_graph":base["pred_test_all"][me],
            "y_pred_final":final[me],"gmodel":base["model"],"hmodel":booster}

def choose_best_tau_dynamic(tr, va, ctx, tau_list):
    best_tau,best_mae=None,np.inf; rows=[]
    for tau in tau_list:
        if tau>=tr["X"].shape[1]: continue
        res=fit_huber_graph(tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau)
        mae=mean_absolute_error(res["y_true"],res["y_pred"]); rows.append({"tau":tau,"val_MAE":mae})
        if mae<best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(tr, va, A_static, tau_list):
    best_tau,best_mae=None,np.inf; rows=[]
    for tau in tau_list:
        if tau>=tr["X"].shape[1]: continue
        m=Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=500))])
        res=fit_static_graph_regressor(m,tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],A_static,tau)
        mae=mean_absolute_error(res["y_true"],res["y_pred"]); rows.append({"tau":tau,"val_MAE":mae})
        if mae<best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params, tau, tr, va, ctx):
    al,ep,lha,md,eta,sub,col,lam,nr,wm,wd_w,wt=params
    hybrid=fit_huber_hybrid(tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau,
        alpha=float(al),eps=float(ep),huber_alpha=10**float(lha),
        xgb_params={"objective":"reg:pseudohubererror","max_depth":int(round(md)),
            "eta":float(eta),"subsample":float(sub),"colsample_bytree":float(col),
            "lambda":float(lam),"tree_method":"hist","seed":RANDOM_SEED,"verbosity":0},
        num_boost_round=int(round(nr)),seed=RANDOM_SEED,
        w_mid=float(wm),w_danger=float(wd_w),w_tail=float(wt))
    mae=mean_absolute_error(hybrid["y_true"],hybrid["y_pred_final"])
    idx=hybrid["y_true"]>=np.percentile(hybrid["y_true"],99)
    if idx.sum()>0:
        mfb,_=compute_mfb_nmse(hybrid["y_true"][idx],hybrid["y_pred_final"][idx])
        return mae+10.0*abs(mfb)
    return mae

def jso_optimize(tr, va, ctx, tau, bounds, n_pop=12, iters=15, seed=1):
    np.random.seed(seed); dim=bounds.shape[0]
    pop=bounds[:,0]+np.random.rand(n_pop,dim)*(bounds[:,1]-bounds[:,0])
    fit=np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in pop])
    best_p=pop[np.argmin(fit)].copy(); best_f=float(fit.min())
    for it in range(iters):
        c=1.0-it/max(iters,1); new_pop=pop.copy()
        for i in range(n_pop):
            if np.random.rand()<0.5:
                cand=pop[i]+np.random.randn(dim)*c*0.1+c*(best_p-pop[i])*np.random.rand(dim)
            else:
                j=np.random.randint(0,n_pop); cand=pop[i]+(pop[j]-pop[i])*(np.random.rand(dim)-0.5)*c
            new_pop[i]=np.clip(cand,bounds[:,0],bounds[:,1])
        nf=np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in new_pop])
        better=nf<fit; pop[better]=new_pop[better]; fit[better]=nf[better]
        if fit.min()<best_f: best_f=float(fit.min()); best_p=pop[np.argmin(fit)].copy()
        print(f"  iter {it+1:02d}/{iters} best={best_f:.4f}")
    return best_p, best_f

def add_result_row(rows, tail_rows, H, split, model, y_true, y_pred, extra=None):
    s=summarize_regression(y_true,y_pred)
    row={"H":H,"split":split,"model":model,"MAE":s["MAE"],"RMSE":s["RMSE"],"R2":s["R2"],"n_test":len(y_true)}
    if extra: row.update(extra)
    rows.append(row)
    for p,thr,mae,rmse,n_tail in compute_tail_metrics(y_true,y_pred):
        tr={"H":H,"split":split,"model":model,"percentile":p,"threshold":float(thr),
            "tail_MAE":float(mae) if pd.notna(mae) else np.nan,
            "tail_RMSE":float(rmse) if pd.notna(rmse) else np.nan,"n_tail":int(n_tail)}
        if extra: tr.update(extra)
        tail_rows.append(tr)

# ============================================================
# CELLS 5-11: DATA LOADING + TRAINING (identical flow)
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)
df = load_beijing_temp()
station_ids = sorted(df["station_id"].unique())
N = len(station_ids)
u_idx = FEATURES.index("u_wind")
v_idx = FEATURES.index("v_wind")

all_times = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")
base = pd.MultiIndex.from_product([all_times, station_ids],
                                   names=["timestamp","station_id"]).to_frame(index=False)
aligned = base.merge(df[["timestamp","station_id"]+FEATURES],
                     on=["timestamp","station_id"], how="left")

X_feat = []
for feat in FEATURES:
    mat = aligned.pivot(index="timestamp", columns="station_id",
                        values=feat).reindex(all_times)[station_ids]
    X_feat.append(mat.to_numpy(dtype=np.float32))
X_all_raw = np.stack(X_feat, axis=-1)

target_raw = aligned.pivot(index="timestamp", columns="station_id",
                            values=TARGET_COL).reindex(all_times)[station_ids].to_numpy(dtype=np.float32)
Y_mask_full = (~np.isnan(target_raw)).astype(np.float32)
fill_values = build_fill_values_from_train_timeline(X_all_raw[all_times < SPLIT_TIME])

print(f"Tensor {X_all_raw.shape}  Target coverage: {Y_mask_full.mean()*100:.1f}%")

# Graph
nodes = build_nodes_from_coords(station_ids, STATION_COORDS)
edges_df = build_edges_from_nodes(nodes, k=GRAPH_K)
A_static = build_static_adj(nodes, k=GRAPH_K)
src = edges_df["src"].to_numpy(dtype=int)
dst = edges_df["dst"].to_numpy(dtype=int)
w_dist = edges_df["w_dist"].to_numpy(dtype=np.float32)
edge_bearing = bearing_radians(nodes.loc[src,"lat"].to_numpy(), nodes.loc[src,"lon"].to_numpy(),
                                nodes.loc[dst,"lat"].to_numpy(), nodes.loc[dst,"lon"].to_numpy())
graph_ctx = {"src":src,"dst":dst,"w_dist":w_dist,"edge_bearing":edge_bearing,
             "u_idx":u_idx,"v_idx":v_idx,"N":N}

# Windowing
def build_windows(X_raw, Y_raw, Y_mask, all_times, split_time, L, H_list):
    n_times = len(all_times); data = {}
    for H in H_list:
        X_wins,Y_wins,M_wins,ttimes = [],[],[],[]
        for t in range(L, n_times-H):
            X_wins.append(X_raw[t-L:t]); Y_wins.append(Y_raw[t+H])
            M_wins.append(Y_mask[t+H]); ttimes.append(all_times[t+H])
        X_arr=np.stack(X_wins); Y_arr=np.stack(Y_wins)
        M_arr=np.stack(M_wins); t_arr=np.array(ttimes)
        test_mask = t_arr >= split_time; train_mask = ~test_mask
        data[H] = {
            "X_train_imp": impute_windows(X_arr[train_mask], fill_values),
            "X_test_imp":  impute_windows(X_arr[test_mask],  fill_values),
            "Y_train":Y_arr[train_mask],"Y_test":Y_arr[test_mask],
            "M_train":M_arr[train_mask],"M_test":M_arr[test_mask],
            "target_times_train":t_arr[train_mask],"target_times_test":t_arr[test_mask],
        }
        print(f"H={H}h  train={train_mask.sum()}  test={test_mask.sum()}")
    return data

print("Building windows...")
all_data = build_windows(X_all_raw, target_raw, Y_mask_full, all_times, SPLIT_TIME, L, H_LIST)

bounds = np.array([[1.0,8.0],[0.01,0.20],[-5.0,-2.0],[3.0,8.0],[0.02,0.15],
                   [0.60,1.00],[0.60,1.00],[0.10,5.0],[200.0,600.0],
                   [1.5,4.0],[3.0,8.0],[6.0,20.0]])

results_rows, tail_rows, tau_rows = [], [], []
all_models = {}

for H in H_LIST:
    print(f"\n{'='*50}\nH={H}h\n{'='*50}")
    pack=all_data[H]
    Xtr=pack["X_train_imp"]; Ytr=pack["Y_train"]; Mtr=pack["M_train"]
    Xte=pack["X_test_imp"];  Yte=pack["Y_test"];  Mte=pack["M_test"]
    tr,va=train_val_split_time_order(Xtr,Ytr,Mtr,pack["target_times_train"],frac=VALID_FRAC)
    btd,_=choose_best_tau_dynamic(tr,va,graph_ctx,TAU_LIST)
    bts,_=choose_best_tau_static(tr,va,A_static,TAU_LIST)
    all_models[H]={"tau_dyn":btd,"tau_static":bts}

    for name,model in [
        ("Ridge",Pipeline([("sc",StandardScaler()),("r",Ridge(alpha=10.0))])),
        ("Huber",Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=1000))])),
        ("RF",RandomForestRegressor(n_estimators=100,max_depth=8,n_jobs=-1,random_state=RANDOM_SEED)),
        ("XGB",XGBRegressor(n_estimators=200,max_depth=8,learning_rate=0.07,subsample=0.9,
                             colsample_bytree=0.8,tree_method="hist",n_jobs=-1,random_state=RANDOM_SEED)),
    ]:
        res=fit_flat_regressor(model,Xtr,Ytr,Mtr,Xte,Yte,Mte)
        add_result_row(results_rows,tail_rows,H,"test",name,res["y_true"],res["y_pred"])

    for name,model in [
        ("SG-Ridge",Pipeline([("sc",StandardScaler()),("r",Ridge(alpha=10.0))])),
        ("SG-Huber",Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=1000))])),
    ]:
        res=fit_static_graph_regressor(model,Xtr,Ytr,Mtr,Xte,Yte,Mte,A_static,bts)
        add_result_row(results_rows,tail_rows,H,"test",name,res["y_true"],res["y_pred"],{"tau":bts})

    res=fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,graph_ctx,tau=btd)
    add_result_row(results_rows,tail_rows,H,"test","Huber-Graph",res["y_true"],res["y_pred"],{"tau":btd})

    res=fit_huber_hybrid(Xtr,Ytr,Mtr,Xte,Yte,Mte,graph_ctx,tau=btd,num_boost_round=400,seed=RANDOM_SEED)
    add_result_row(results_rows,tail_rows,H,"test","Hybrid",res["y_true"],res["y_pred_final"],{"tau":btd})

    if RUN_TUNED:
        best_p,best_obj=jso_optimize(tr,va,graph_ctx,btd,bounds,JSO_POP,JSO_ITERS,RANDOM_SEED)
        al,ep,lha,md,eta,sub,col,lam,nr,wm,wd_w,wt=best_p; ha=10**float(lha)
        res=fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,graph_ctx,tau=btd,alpha=float(al),eps=float(ep),huber_alpha=ha)
        add_result_row(results_rows,tail_rows,H,"test","Tuned-Huber-Graph",res["y_true"],res["y_pred"],{"tau":btd,"val_obj":float(best_obj)})
        res=fit_huber_hybrid(Xtr,Ytr,Mtr,Xte,Yte,Mte,graph_ctx,tau=btd,alpha=float(al),eps=float(ep),huber_alpha=ha,
            xgb_params={"objective":"reg:pseudohubererror","max_depth":int(round(md)),"eta":float(eta),
                "subsample":float(sub),"colsample_bytree":float(col),"lambda":float(lam),
                "tree_method":"hist","seed":RANDOM_SEED,"verbosity":0},
            num_boost_round=int(round(nr)),seed=RANDOM_SEED,w_mid=float(wm),w_danger=float(wd_w),w_tail=float(wt))
        add_result_row(results_rows,tail_rows,H,"test","Tuned-Hybrid",res["y_true"],res["y_pred_final"],{"tau":btd,"val_obj":float(best_obj)})
    print(f"H={H} done")

results_df=pd.DataFrame(results_rows).sort_values(["H","MAE"]).reset_index(drop=True)
tail_df=pd.DataFrame(tail_rows).sort_values(["H","model","percentile"]).reset_index(drop=True)
results_df.to_csv(f"{OUT_DIR}/results_main.csv",index=False)
tail_df.to_csv(f"{OUT_DIR}/results_tail.csv",index=False)
joblib.dump(all_models,f"{OUT_DIR}/models.pkl")
print("\n=== TEMPERATURE FORECASTING RESULTS ==="); display(results_df)


Loaded 420,768 rows, 12 stations
Tensor (35064, 12, 5)  Target coverage: 99.9%
Building windows...
H=1h  train=30695  test=4344
H=3h  train=30693  test=4344
H=6h  train=30690  test=4344
H=12h  train=30684  test=4344
H=24h  train=30672  test=4344

H=1h
  iter 01/8 best=1.1499
  iter 02/8 best=1.1414
  iter 03/8 best=1.1414
  iter 04/8 best=1.1414
  iter 05/8 best=1.1414
  iter 06/8 best=1.1414
  iter 07/8 best=1.1390
  iter 08/8 best=1.1390
H=1 done

H=3h
  iter 01/8 best=1.9366
  iter 02/8 best=1.9041
  iter 03/8 best=1.9041
  iter 04/8 best=1.9041
  iter 05/8 best=1.9041
  iter 06/8 best=1.9041
  iter 07/8 best=1.8961
  iter 08/8 best=1.8932
H=3 done

H=6h
  iter 01/8 best=2.8424
  iter 02/8 best=2.8371
  iter 03/8 best=2.7971
  iter 04/8 best=2.7971
  iter 05/8 best=2.7971
  iter 06/8 best=2.7971
  iter 07/8 best=2.7971
  iter 08/8 best=2.7971
H=6 done

H=12h
  iter 01/8 best=3.3658
  iter 02/8 best=3.3658
  iter 03/8 best=3.3658
  iter 04/8 best=3.3607
  iter 05/8 best=3.3607
  iter

,H,split,model,MAE,RMSE,R2,n_test,tau,val_obj
0,1,test,XGB,0.915002,1.246932,0.982027,51909,NaN,NaN
1,1,test,Huber,0.933278,1.312510,0.980087,51909,NaN,NaN
2,1,test,Ridge,0.943247,1.292048,0.980703,51909,NaN,NaN
3,1,test,Hybrid,0.989337,1.376604,0.978094,51909,2.0,NaN
4,1,test,Tuned-Hybrid,0.996605,1.389889,0.977669,51909,2.0,1.138993
5,1,test,RF,1.092889,1.473034,0.974918,51909,NaN,NaN
6,1,test,SG-Huber,1.214490,1.717896,0.965886,51909,2.0,NaN
7,1,test,Tuned-Huber-Graph,1.215578,1.719570,0.965820,51909,2.0,1.138993
8,1,test,Huber-Graph,1.218924,1.723192,0.965675,51909,2.0,NaN
9,1,test,SG-Ridge,1.242748,1.696982,0.966712,51909,2.0,NaN
